# Native quantum programming tutorial, part 1

All original teaching stages have native Python solutions. Exercise placeholders are explicitly completed here; the deliberately invalid ownership example remains an expected synthesis refusal. Classical arguments bind at synthesis time, so changing alpha or beta requires synthesizing again. The original signed three-bit register, eight probabilities, full-turn rotation convention, arithmetic widths and nonzero accumulator are preserved.

Array index zero is the low bit; Circuit comparisons explicitly account for wire order. No authentication, hosted IDE or external execution is required.

## Local setup

Original code cell 2. Run locally and inspect the actual result.

In [ ]:
import math
from typing import Annotated

from examples.synthesis.tutorials.basics.quantum_programming.qmod_tutorial_part1.overview.workflow import (
    initialization_refusal,
    sample,
    signed_distribution,
    verify_program,
)
from flagquantum import synth
from flagquantum.drawer import draw
from flagquantum.synth import (
    CX,
    RX,
    H,
    Output,
    QArray,
    QBit,
    QNum,
    X,
    allocate,
    control,
    hadamard_transform,
    inplace_prepare_state,
    prepare_state,
    qfunc,
)

checks = []
observations = {}

## Bell preparation

Original code cell 5. Run locally and inspect the actual result.

In [ ]:
@qfunc
def bell_helper(pair: QArray[QBit]) -> None:
    H(pair[0])
    CX(pair[0], pair[1])

@qfunc
def bell(res: Output[QArray[QBit]]) -> None:
    allocate(2, res)
    bell_helper(res)

program = synth.synthesize(bell, memory_limit_bytes=64 * 1024**2)
check = verify_program("bell", program)
checks.append(check)
print(draw(program.circuit))
observations["bell"] = sample(program, shots=1000)
print(observations["bell"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)

## Exercise A reference solution

Original code cell 9. The original TODO is replaced by its supplied solution; this is not credited as a distinct algorithm.

In [ ]:
@qfunc
def ghz_helper(qubits: QArray[QBit]) -> None:
    H(qubits[0])
    CX(qubits[0], qubits[1])
    CX(qubits[1], qubits[2])

@qfunc
def ghz(res: Output[QArray[QBit]]) -> None:
    allocate(3, res)
    ghz_helper(res)

program = synth.synthesize(ghz, memory_limit_bytes=64 * 1024**2)
check = verify_program("ghz", program)
checks.append(check)
print(draw(program.circuit))
observations["ghz"] = sample(program, shots=1000)
print(observations["ghz"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)

## Signed numeric view reference solution

Original code cell 12. The original TODO is replaced by its supplied solution; this is not credited as a distinct algorithm.

In [ ]:
@qfunc
def signed_ghz(x: Output[Annotated[QNum, (3, True, 0)]]) -> None:
    allocate(x)
    ghz_helper(x)

program = synth.synthesize(signed_ghz, memory_limit_bytes=64 * 1024**2)
check = verify_program("signed_ghz", program)
checks.append(check)
print(draw(program.circuit))
observations["signed_ghz"] = sample(program, shots=1000)
print(observations["signed_ghz"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)
print("Signed numeric values:", dict(signed_distribution(program)))

## Probability preparation reference solution

Original code cell 16. The original TODO is replaced by its supplied solution; this is not credited as a distinct algorithm.

In [ ]:
probabilities = [0.3, 0.2, 0.05, 0.13, 0.15, 0.05, 0.1, 0.02]

@qfunc
def distributions(
    probabilities: list[float], a: Output[QArray[QBit]], b: Output[QArray[QBit]]
) -> None:
    prepare_state(probabilities, 0.01, a)
    allocate(3, b)
    inplace_prepare_state(probabilities, 0.01, b)

program = synth.synthesize(distributions, classical_inputs={"probabilities": probabilities}, memory_limit_bytes=64 * 1024**2)
check = verify_program("distributions", program)
checks.append(check)
print(draw(program.circuit))
observations["distributions"] = sample(program, shots=10000)
print(observations["distributions"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)

## Expected initialization refusal

Original code cell 19. Run locally and inspect the actual result.

In [ ]:
print(initialization_refusal())

## Rotation reference solution

Original code cell 22. The original TODO is replaced by its supplied solution; this is not credited as a distinct algorithm.

In [ ]:
@qfunc
def rotate(turns: float, index: int, qarr: QArray[QBit]) -> None:
    RX(2 * math.pi * turns, qarr[index])

@qfunc
def rotations(qarr: Output[QArray[QBit]]) -> None:
    allocate(2, qarr)
    rotate(0.3, 0, qarr)
    rotate(0.2, 1, qarr)

program = synth.synthesize(rotations, memory_limit_bytes=64 * 1024**2)
check = verify_program("rotations", program)
checks.append(check)
print(draw(program.circuit))
observations["rotations"] = sample(program, shots=1000)
print(observations["rotations"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)

## Parameter binding reference solution

Original code cell 25. The original TODO is replaced by its supplied solution; this is not credited as a distinct algorithm.

In [ ]:
@qfunc
def parameterized(alpha: float, beta: float, qarr: Output[QArray[QBit]]) -> None:
    allocate(2, qarr)
    rotate(alpha, 0, qarr)
    rotate(beta, 1, qarr)

program = synth.synthesize(parameterized, classical_inputs={"alpha": 0.3, "beta": 0.2}, memory_limit_bytes=64 * 1024**2)
check = verify_program("parameterized", program)
checks.append(check)
print(draw(program.circuit))
observations["parameterized"] = sample(program, shots=1000)
print(observations["parameterized"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)

## Array helper reference solution

Original code cell 28. The original TODO is replaced by its supplied solution; this is not credited as a distinct algorithm.

In [ ]:
@qfunc
def rotate_layer(turns: list[float], qarr: QArray[QBit]) -> None:
    for index in range(len(turns)):
        rotate(turns[index], index, qarr)

@qfunc
def layer(qarr: Output[QArray[QBit]]) -> None:
    allocate(3, qarr)
    rotate_layer([0.1, 0.3, 0.5], qarr)

program = synth.synthesize(layer, memory_limit_bytes=64 * 1024**2)
check = verify_program("layer", program)
checks.append(check)
print(draw(program.circuit))
observations["layer"] = sample(program, shots=1000)
print(observations["layer"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)

## Arithmetic reference solution

Original code cell 31. The original TODO is replaced by its supplied solution; this is not credited as a distinct algorithm.

In [ ]:
@qfunc
def arithmetic(
    x: Output[QNum], y: Output[QNum], total: Output[QNum], prod: Output[QNum]
) -> None:
    allocate(2, x)
    allocate(2, y)
    hadamard_transform(x)
    hadamard_transform(y)
    total |= x + y
    prod |= x * y

program = synth.synthesize(arithmetic, memory_limit_bytes=64 * 1024**2)
check = verify_program("arithmetic", program)
checks.append(check)
print(draw(program.circuit))
observations["arithmetic"] = sample(program, shots=1000)
print(observations["arithmetic"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)

## In-place addition reference solution

Original code cell 34. The original TODO is replaced by its supplied solution; this is not credited as a distinct algorithm.

In [ ]:
@qfunc
def accumulation(y: Output[QNum], acc: Output[QNum]) -> None:
    allocate(2, y)
    hadamard_transform(y)
    acc |= 4
    acc += y

program = synth.synthesize(accumulation, memory_limit_bytes=64 * 1024**2)
check = verify_program("accumulation", program)
checks.append(check)
print(draw(program.circuit))
observations["accumulation"] = sample(program, shots=1000)
print(observations["accumulation"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)

## Coherent condition reference solution

Original code cell 38. The original TODO is replaced by its supplied solution; this is not credited as a distinct algorithm.

In [ ]:
@qfunc
def conditional(x: Output[QNum], target: Output[QBit]) -> None:
    allocate(3, x)
    hadamard_transform(x)
    allocate(target)
    control(x == 5, lambda: X(target))

program = synth.synthesize(conditional, memory_limit_bytes=64 * 1024**2)
check = verify_program("conditional", program)
checks.append(check)
print(draw(program.circuit))
observations["conditional"] = sample(program, shots=1000)
print(observations["conditional"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)

## Exercise A solution

Original code cell 43. Run locally and inspect the actual result.

In [ ]:


program = synth.synthesize(ghz, memory_limit_bytes=64 * 1024**2)
check = verify_program("ghz", program)
checks.append(check)
print(draw(program.circuit))
observations["ghz"] = sample(program, shots=1000)
print(observations["ghz"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)

## Signed solution

Original code cell 45. Run locally and inspect the actual result.

In [ ]:

program = synth.synthesize(signed_ghz, memory_limit_bytes=64 * 1024**2)
check = verify_program("signed_ghz", program)
checks.append(check)
print(draw(program.circuit))
observations["signed_ghz"] = sample(program, shots=1000)
print(observations["signed_ghz"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)
print("Signed numeric values:", dict(signed_distribution(program)))

## Preparation solution

Original code cell 48. Run locally and inspect the actual result.

In [ ]:

program = synth.synthesize(distributions, classical_inputs={"probabilities": probabilities}, memory_limit_bytes=64 * 1024**2)
check = verify_program("distributions", program)
checks.append(check)
print(draw(program.circuit))
observations["distributions"] = sample(program, shots=10000)
print(observations["distributions"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)

## Ownership correction

Original code cell 50. Run locally and inspect the actual result.

In [ ]:
@qfunc
def owned_prep(q: Output[QBit]) -> None:
    allocate(q)
    H(q)

@qfunc
def corrected_initialization(q: Output[QBit]) -> None:
    owned_prep(q)

program = synth.synthesize(corrected_initialization, memory_limit_bytes=64 * 1024**2)
check = verify_program("corrected_initialization", program)
checks.append(check)
print(draw(program.circuit))
observations["corrected_initialization"] = sample(program, shots=1000)
print(observations["corrected_initialization"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)

## Turns solution

Original code cell 53. Run locally and inspect the actual result.

In [ ]:


program = synth.synthesize(rotations, memory_limit_bytes=64 * 1024**2)
check = verify_program("rotations", program)
checks.append(check)
print(draw(program.circuit))
observations["rotations"] = sample(program, shots=1000)
print(observations["rotations"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)

## Parameter solution

Original code cell 55. Run locally and inspect the actual result.

In [ ]:

program = synth.synthesize(parameterized, classical_inputs={"alpha": 0.3, "beta": 0.2}, memory_limit_bytes=64 * 1024**2)
check = verify_program("parameterized", program)
checks.append(check)
print(draw(program.circuit))
observations["parameterized"] = sample(program, shots=1000)
print(observations["parameterized"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)

## Layer solution

Original code cell 57. Run locally and inspect the actual result.

In [ ]:


program = synth.synthesize(layer, memory_limit_bytes=64 * 1024**2)
check = verify_program("layer", program)
checks.append(check)
print(draw(program.circuit))
observations["layer"] = sample(program, shots=1000)
print(observations["layer"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)

## Arithmetic solution

Original code cell 60. Run locally and inspect the actual result.

In [ ]:

program = synth.synthesize(arithmetic, memory_limit_bytes=64 * 1024**2)
check = verify_program("arithmetic", program)
checks.append(check)
print(draw(program.circuit))
observations["arithmetic"] = sample(program, shots=1000)
print(observations["arithmetic"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)

## Accumulator solution

Original code cell 62. Run locally and inspect the actual result.

In [ ]:

program = synth.synthesize(accumulation, memory_limit_bytes=64 * 1024**2)
check = verify_program("accumulation", program)
checks.append(check)
print(draw(program.circuit))
observations["accumulation"] = sample(program, shots=1000)
print(observations["accumulation"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)

## Condition solution

Original code cell 64. Run locally and inspect the actual result.

In [ ]:

program = synth.synthesize(conditional, memory_limit_bytes=64 * 1024**2)
check = verify_program("conditional", program)
checks.append(check)
print(draw(program.circuit))
observations["conditional"] = sample(program, shots=1000)
print(observations["conditional"])
print("Absolute state error:", check.maximum_error, "Dirty work:", check.dirty_workspace_probability)